# ASSIGNMENT 5 INSTRUCTIONS

In class, we worked through a mini-case in which we mapped the as-is return and refund process for a mid-sized e-commerce company using a swimlane diagram. We then analyzed the process to identify key inefficiencies and determined that the Customer Service Representative role could be augmented - or partially automated - using an agentic AI workflow. In this assignment, we will design and implement that workflow and test it using a synthetic database of customer data.

**This assignment accomplishes the following objectives:**
1. Internalize the end-to-end process introduced in class: analyzing an as-is process, identifying opportunities for AI augmentation, designing the to-be process, and implementing the corresponding agentic AI workflow.
2. Learn how to run Gradio in chat mode.
3. Understand why evaluations are essential for agentic AI workflows, and learn how to use an LLM-as-a-judge approach to assess non-deterministic outputs.

**In order to complete the assignment:**
1. Section 1: Assignment setup. Just read through to understand what's already available.
2. Section 2: Create the agenticAI workflow. Complete coding steps 2.1 through 2.4.
3. Section 3: Execute and Test your workflow using both the Gradio UI and evaluation function.

**To submit:**
1. Rename the assignment as Assignment5-\<asurite\>
2. Runtime -> Restart session and run all
3. Ctrl-s
4. File -> Download .ipynb
5. Submit on Canvas

# SECTION 1: ASSIGNMENT SETUP (JUST READ THROUGH)

### We'll first import the required libraries.

In [1]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# install the orchestration layer (aisuite) plus the provider SDKs (OpenAI, Groq)
# in Colab, this is typically needed once per new runtime
!pip install aisuite openai groq
!pip install --upgrade gradio

### Next, we'll setup the API KEYS and Clients.

In [2]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import os
import aisuite
from openai import OpenAI
from groq import Groq

# load API KEYS (Google Colab + local env fallback)
try:
  # read the API KEYs from Colab Secrets and expose it as an env variable
  from google.colab import userdata
  os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
  # read the API KEYs from local env and expose it as an env variable
  os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")
  os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")

# create the clients
openai_client = OpenAI()
groq_client = Groq()
aisuite_client = aisuite.Client()

# select the models you want to use
OPENAI_MODEL= "gpt-4.1-mini"
AISUITE_OPENAI_MODEL = "openai:"+OPENAI_MODEL
GROQ_MODEL = "llama-3.3-70b-versatile"
AISUITE_GROQ_MODEL = "groq:"+GROQ_MODEL


### Next, we'll set the sqlite_db_path for the customer database.

In [3]:
#######################################################
##### DO NOT EDIT THIS CELL - except to update filepath
#######################################################

# import sqlite3
# import pandas as pd

# set sqlite db path (Google Colab + local fallback)
sqlite_db_path=""
try:
    from google.colab import drive
except ImportError:
    sqlite_db_path = "agentic_returns_demo_10users.db"
else:
    drive.mount('/content/drive')
    sqlite_db_path = "/content/drive/MyDrive/agentic_returns_demo_10users.db"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### As before, we'll use this utility function to format LLM responses.

In [4]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# a nifty utility function to cleanly diplay your chat outputs
from IPython.display import display, HTML, Markdown

def show(title, text):
    display(HTML(f"""
     <h4>{title}</h4>
     <div style="white-space: pre-wrap; padding-left: 24px;">{text}</div>"""))

### As before, we'll provide the OpenAI LLM access to OpenAI's built-in web_search tool.

In [5]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import json

# use OpenAI's built-in web_search tool to perform web_search
def openai_web_search(query: str):
  """
    Perform a web search using OpenAI's built-in web_search tool
    and return raw results.
  """
  resp = openai_client.responses.create(
      model=OPENAI_MODEL,
      input=query,
      tools=[{"type": "web_search"}],
  )
  return json.loads(resp.model_dump_json())


### As before, the run_openai_query accepts the system prompt, the user prompt, and a list of tools that it can invoke as needed. And it returns both the model's response and relevant query details (including usage statistics and any tools that were called).

In [6]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

# run the system/user prompt using OpenAI API, invoke tools as needed;
# return response and query details (usage stats and tools invoked)
def run_openai_query(system_prompt, user_prompt, tools=[]):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Supports tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to OpenAI-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_OPENAI_MODEL, # OpenAI-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      tools=tools,
      max_turns=3,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details


### As before, the run_groq_query is similar to the run_openai_query, with one important difference: it does not accept/invoke any tools. It returns both the model's response and relevant query details (including usage statistics and any tools that were called).

In [7]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

# run the system/user prompt using OpenAI API; DOES NOT support tools;
# return response and query details (usage stats and tools invoked)
def run_groq_query(system_prompt, user_prompt):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Does not support tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to Groq-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_GROQ_MODEL, # Groq-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      max_turns=1,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details

# SECTION 2: IMPLEMENT THE AGENTIC-AI WORKFLOW (COMPLETE STEPS 2.1 THROUGH 2.4)

Recall, we analyzed an as-is process in class for a mini-case, identified opportunities for AI augmentation, and designed the to-be process. We're now ready to implement the corresponding agentic AI workflow.

- user_message → **guardrail_agent** → ALLOW or BLOCK:\<reason\>
- user_message, policy_text → **policy_agent** → POLICY_ANSWER:\<answer\>
- user_message, user_id, sqlite_db_path → **database_agent** → DATABASE_SUMMARY:\{table_name: \[rows...\]\} [ALL rows for user_id]
- user_message, policy_text, database_summary → **return_decision_agent** → DECISION:REFUND_WITHOUT_RETURN, or, DECISION:RETURN_REQUIRED, or, DECISION:NEED_MORE_INFO:\<reason\>
- user_message, policy_answer, database_summary → **escalation_agent** → NO_ESCALATION, or, ESCALATE:\<reason\>
- user_message, policy_answer, database_summary decision, escalation → **response_composer_agent** → \<reponse to user\>

> **NOTE:** You may need to iterate on these steps multiple times before settling on the final version for submission.

### For this assignment, we're going to capture the company policy for returns and refunds in a simple string. In a production system, this information might sit in a database.

In [8]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# returns and refunds policy
policy_text = (
  f"""
  RETURNS & REFUNDS POLICY

  1. Return Window
  - Customers may request a return within 30 days of delivery.
  - Returns requested after 30 days are not eligible unless approved by a supervisor.

  2. Final Sale Items
  - Final sale items are not eligible for return.
  - Exception: Final sale items may be refunded if damaged or if the wrong item was sent.

  3. Refund Without Return
  - Items priced at $25 or less may be refunded without requiring the item to be returned.
  - Items above $25 require the item to be shipped back before refund is issued.

  4. Return Shipping Labels
  - If a return is required, we provide a prepaid return shipping label.
  - Customers must use the provided label so the return can be tracked.

  5. Damaged / Wrong Item Claims (Proof Required)
  - For damaged items or wrong item shipments, customers must provide a photo.
  - If no photo is provided, the case may require supervisor review or may be denied.

  6. Inspection (For Returned Items)
  - Returned items are inspected when received.
  - If inspection fails (e.g., missing components, signs of heavy use, damage not reported),
    the refund may be denied or reduced.

  7. Refund Method
  - Approved refunds are issued to the original payment method used on the order.
  - Store credit may be offered if the original payment method cannot be refunded.

  8. Refund Timing
  - Refunds are processed after return inspection is complete (if a return is required).
  - Once processed, refunds typically post within 3–5 business days.
  - During high-volume periods, posting may take up to 10 business days.

  9. Escalation / Supervisor Review
  - Refunds over $150 require supervisor approval.
  - Customers with repeated refund patterns may require supervisor review.
  - Disputes involving final sale items, missing proof, or failed inspection require supervisor review.
  """
)

### [Complete Coding STEP 2.1]:

You'll now define the **guardrail** agent: <br>
user_message → guardrail_agent → ALLOW or BLOCK:\<reason\>

The Guardrail Agent blocks requests for other customer's data, bulk exports, or internal database access. It allows policy questions and the customer’s own order questions.

You'll update the system_prompt (STEP 2.1) as you see fit to accomplish this.

In [9]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# returns ALLOW or BLOCK:<reason>
def guardrail_agent(user_message: str) -> str:

  # STEP-2.1: You have been provided with a starter system prompt for the guardrail agent.
  #           Populate the "Block" and "Allow" sections with clear, specific criteria that
  #           determine which types of user requests should be blocked versus permitted.
  #           In general, users are NOT PERMITTED to access data of other users,
  #           and they are NOT PERMITTED to directly edit data belonging to any user
  #           (including themselves).
  #           Your criteria should reflect:
  #           - Data privacy considerations
  #           - Data modification or system manipulation risks
  #           - Legitimate customer service inquiries
  guardrail_system_prompt = (
    f"""
    You are a guardrail agent.

    Block:
      - Requests for order, refund, or personal information belonging to anyone other than the authenticated user (e.g., a spouse or friend).
      - Requests to execute bulk data exports.
      - Any attempts to directly edit, modify, or delete database records.

    Allow:
      - Inquiries regarding the store's official returns and refunds policy.
      - Questions specifically concerning the authenticated customer's own order history, refund status, or return eligibility.

    Return ONLY:
      ALLOW
      or
      BLOCK: <short reason>
    """
  )

  # user prompt: do not edit
  guardrail_user_prompt = (
    f"""
    User message: {user_message}
    Return only ALLOW or BLOCK: ...
    """
  )

  resp_text, _ = run_openai_query(guardrail_system_prompt, guardrail_user_prompt)
  return resp_text.strip().upper()

### [Complete Coding STEP 2.2]:

You'll now define the **policy** agent: <br>
user_message, policy_text → policy_agent → POLICY_ANSWER:\<answer\>

The Policy Agent uses only the policy text to answer eligibility questions or to extract relevant rules (return window, final sale exception, photo requirement, refund‑without‑return threshold).

You'll update the system_prompt (STEP 2.2) to accomplish this.

In [10]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# returns POLICY_ANSWER:<text>
def policy_agent(user_message: str, policy_text: str) -> str:

  # STEP-2.2: You have been provided with a starter system prompt for the policy agent.
  #           Populate the "Rules" section with explicit rules that:
  #           - Classify whether a user request requires consulting policy text.
  #           - Determine when no policy lookup is needed.
  #           - Handle cases where the policy text does not contain enough information.
  policy_system_prompt = (
    f"""
    You are a policy agent.
    Use ONLY the provided policy text.

    Rules:
      - If the user asks about eligibility, return windows, or exceptions, extract the exact corresponding rule from the policy text.
      - If the policy text does not explicitly contain the information requested (e.g., exact bank posting timelines), state that the information is not specified; do not invent or guess an answer.
      - If the user's message is a general greeting or does not pertain to returns/refunds, state that no policy lookup is needed.

    Return ONLY: POLICY_ANSWER: <answer>
    """
  )

  # user prompt: do not edit
  policy_user_prompt = (
    f"""
    User question: {user_message}
    Policy text:{policy_text}

    Return only POLICY_ANSWER: ...
    """
  )

  resp_text, _ = run_openai_query(policy_system_prompt, policy_user_prompt)
  return resp_text.strip()


We'll now define the **database** agent: <br>
user_message, user_id, sqlite_db_path → database_agent → DATABASE_SUMMARY:{table_name: [rows...]}

The Database Agent looks up the authenticated customer’s transaction history.

Notice that we've implemented a very simple database_agent that simply queries the sqlite database and returns **all** rows for the given customer. This would never fly in a productin system with a large number of customers and a large number of transactions per customer. But it works for the purposes of this assignment.

In [11]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# very simple db agent that simply returns *all* rows for given customer
# returns {"DATABASE_SUMMARY": {table_name: [rows...]}}
def database_agent(user_message: str, user_id: str, sqlite_db_path: str):

  import sqlite3

  tables = [
    "customers",
    "orders",
    "order_items",
    "return_requests",
    "return_shipments",
    "warehouse_inspections",
    "refunds",
    "tickets",
    "ticket_messages",
  ]

  try:
    with sqlite3.connect(f"file:{sqlite_db_path}?mode=ro", uri=True) as conn:
      conn.row_factory = sqlite3.Row
      data = {}
      for t in tables:
        rows = conn.execute(f"SELECT * FROM {t} WHERE customer_id=?", (user_id,)).fetchall()
        data[t] = [dict(r) for r in rows]
    return {"DATABASE_SUMMARY": data}

  except Exception as e:
    return {"DATABASE_SUMMARY": str(e)}


We'll now define the **return_decision** agent: <br>
user_message, policy_text, database_summary → return_decision_agent → DECISION:REFUND_WITHOUT_RETURN, or, DECISION:RETURN_REQUIRED, or, DECISION:NEED_MORE_INFO:\<reason\>

The Return Decision Agent chooses one of: REFUND_WITHOUT_RETURN vs RETURN_REQUIRED vs NEED_MORE_INFO (and lists what’s missing).


In [12]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# returns DECISION:REFUND_WITHOUT_RETURN, DECISION:RETURN_REQUIRED, or DECISION:NEED_MORE_INFO:<reason>
def return_decision_agent(user_message: str, policy_text: str, database_summary: str) -> str:

  return_decision_system_prompt = (
    f"""
    You are a returns decision agent.

    Decide whether this case should be:
      - REFUND_WITHOUT_RETURN (no return needed)
      - RETURN_REQUIRED (customer must ship item back)

    Use the policy text and the database summary.

    Return ONLY:
      DECISION: REFUND_WITHOUT_RETURN
      or
      DECISION: RETURN_REQUIRED
      or
      DECISION: NEED_MORE_INFO: <what is missing>
    """
  )

  return_decision_user_prompt = (
    f"""
    User message: {user_message}
    Policy text: {policy_text}
    Database summary: {database_summary}

    Return only the DECISION line.
    """
  )

  resp_text, _ = run_openai_query(return_decision_system_prompt, return_decision_user_prompt)
  return resp_text.strip()


### [Complete Coding STEP 2.3]:

You'll now define the **escalation** agent: <br>
user_message, policy_answer, database_summary → escalation_agent → NO_ESCALATION, or, ESCALATE:\<reason\>

The Escalation Agent decides whether to escalate to a supervisor (high refund amount>$150, final sale dispute, repeated refunds, missing required proof).

You'll update the system_prompt (STEP 2.3) to accomplish this.

In [13]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# return ESCALATE:<reason> or NO_ESCALATION
def escalation_agent(user_message: str, policy_answer: str, database_summary: str) -> str:

  # STEP-2.3: You have been provided with a starter system prompt for the escalation agent.
  #           Populate the "Escalate if" section with explicit, well-defined conditions that
  #           reflect policy_text.
  escalation_system_prompt = (
    f"""
    You are an escalation agent.

    Escalate if:
      - The refund amount being requested or processed exceeds $150.
      - The customer is requesting a return or disputing an item marked as "Final Sale".
      - The database summary shows the customer has a pattern of repeated refunds.
      - The user is making a claim for a damaged or wrong item, but the required photo proof is missing from the database summary.
      - The return request is made outside the standard 30-day window after delivery.

    Return ONLY:
      ESCALATE: <reason>
      or
      NO_ESCALATION
    """
  )

  # user prompt: do not edit
  escalation_user_prompt = (
    f"""
    User message: {user_message}
    Policy answer: {policy_answer}
    Database summary: {database_summary}

    Return only ESCALATE or NO_ESCALATION.
    """
  )

  resp_text, _ = run_openai_query(escalation_system_prompt, escalation_user_prompt)
  return resp_text.strip()


### [Complete Coding STEP 2.4]:

You'll now define the **response_composer** agent: <br>
user_message, policy_answer, database_summary decision, escalation → response_composer_agent → \<reponse to user\>

The Response Composer writes the customer‑facing response with clear next steps.

You'll update the system_prompt (STEP 2.4) to accomplish this.

In [14]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# return final customer-facing response
def response_composer_agent(user_message: str, policy_answer: str, database_summary: str, decision: str, escalation: str) -> str:

  # STEP-2.4: You have been provided with a starter system prompt for the response_composer agent.
  #           Populate the "Rules" section with clear guidelines that govern how the final
  #           customer-facing message should be constructed.
  response_composer_system_prompt = (
    f"""
    You are a customer service response agent.

    Goal: Respond ONLY to the user's latest message.

    Use:
      - User message
      - Policy answer
      - Database summary
      - Return decision
      - Escalation decision

    Rules:
      - Draft a polite, helpful, and direct response that answers the user's specific query.
      - Clearly explain the return decision (e.g., whether the item requires a return or qualifies for a refund without return) based on the "Return decision" input.
      - State specific facts about the customer's order using the "Database summary" (e.g., order status, item price, tracking numbers).
      - If the "Escalation decision" indicates ESCALATE, politely inform the customer that their case requires and has been sent for supervisor review.
      - Provide explicit next steps, such as instructions to use a provided return shipping label, upload a photo, or await review.
      - NEVER output raw internal system labels (like "DECISION: REFUND_WITHOUT_RETURN", "NEED_MORE_INFO", or "ESCALATE") in the final text.

      Return ONLY the final response text.
    """
  )

  # user prompt: do not edit
  response_composer_user_prompt = (
    f"""
    User message: {user_message}
    Policy answer: {policy_answer}
    Database summary: {database_summary}
    Return decision: {decision}
    Escalation decision: {escalation}
    """
  )

  resp_text, _ = run_openai_query(response_composer_system_prompt, response_composer_user_prompt)
  return resp_text.strip()


### Now that we have all the individual agents defined, we to define the agenticAI workflow.

In [15]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

def agentic_workflow(user_id: str, user_message: str, sqlite_db_path: str, policy_text: str):

  # 1. guardrails
  guard = guardrail_agent(user_message)
  if guard.startswith("BLOCK"):
    return f"Sorry — I can’t help with that. {guard}"

  # 2. policy lookup
  policy_answer = policy_agent(user_message, policy_text)

  # 3. database lookup
  database_summary = database_agent(user_message, user_id, sqlite_db_path)

  # 4. return decision
  decision = return_decision_agent(user_message, policy_text, database_summary)

  # 5. escalation
  escalation = escalation_agent(user_message, policy_answer, database_summary)

  # 6. final response
  final_answer = response_composer_agent(
      user_message,
      policy_answer,
      database_summary,
      decision,
      escalation
  )

  return final_answer


# SECTION 3: EXECUTE AND TEST THE AGENTIC-AI WORKFLOW FOR VARIOUS USERS IN THE CUSTOMER DATABASE

In this section, you will execute and test the agentic AI workflow using various users from the Customer Database. You will do this in two ways:

1. Through the **Gardio UI**
2. Through the **Evaluation Function** *(located in the last cell after the Gradio UI - scroll to the bottom of the notebook)*

Your goal is to refine the prompts you created in Steps 2.1 through 2.4 so that the agentic AI workflow produces high-quality responses. The prompts should be as precise and well-structured as possible to improve performance, while still generalizing effectively to new users and unseen test cases.

**Do not hard-code any specific answers, user details, or case-specific logic in Steps 2.1 through 2.4 to achieve perfect evaluation results**. The objective is to improve the workflow through better prompt design—not by tailoring it to individual test cases.

### (1) Test/Execute Workflow using the Gradio UI
- The Gradio UI provides a dropdown menu of all users in the database.
- You will select a user, pretend to be them, and interact with the customer service bot accordingly.
- You can reset the session at any time by clicking the “Clear / Start Over” button.  
- Test the the workflow through the UI for different users and different customer service queries. If you notice unexpected responses, tweak the workflow system prompts in coding steps 2.1 through 2.4 and try again.

In [16]:
import gradio as gr, sqlite3

# retrieve customer records from the SQLite database
# and format them as (label, value) pairs for the dropdown menu.
def customer_choices(db_path):
    try:
        # open database in read-only mode and fetch customer list
        with sqlite3.connect(f"file:{db_path}?mode=ro", uri=True) as con:
            rows = con.execute(
                "SELECT customer_id, full_name, segment FROM customers ORDER BY customer_id"
            ).fetchall()
        # return dropdown choices as (display_label, actual_value)
        return [(f"{cid} — {name} ({seg})", cid) for cid, name, seg in rows]  # (label, value)
    except Exception:
        # if database connection fails, return empty dropdown
        return []

# initialize the chat session after a customer is selected.
# displays a greeting and disables further changes to the dropdown.
def on_select(cid):
    if not cid:
        return [], gr.update(interactive=True)
    return [{"role": "assistant", "content": f"Hi! You're chatting as **{cid}**."}], gr.update(interactive=False)

# handle user message submission.
# calls the agentic workflow and appends the response to chat history.
def on_chat(text, cid, history):

    # prevent chatting before selecting a customer
    if not cid:
        return "", history or [{"role": "assistant", "content": "Select a pretend customer first."}]

    # initialize history if first message
    history = history or [{"role": "assistant", "content": f"Hi! You're chatting as **{cid}**."}]

    # call agentic workflow to generate response
    reply = agentic_workflow(user_id=cid, user_message=text, sqlite_db_path=sqlite_db_path, policy_text=policy_text)

    # append user message and assistant reply to chat history
    history += [{"role": "user", "content": text}, {"role": "assistant", "content": reply}]
    return "", history

# reset the entire session state.
# clears chat history, re-enables the dropdown, and empties the message box.
def on_clear():
    return [], gr.update(value=None, interactive=True), ""

# create the main Gradio application container and assemble all UI components
# (dropdown, chatbot, textbox, and control buttons).
with gr.Blocks(title="Agentic Returns Demo") as demo:

    # dropdown to select a pretend customer
    cid = gr.Dropdown(customer_choices(sqlite_db_path), label="Pretend customer", value=None)
    # chat window to display conversation
    chat = gr.Chatbot(height=150)
    # text input for user messages
    msg = gr.Textbox(label="Message", placeholder="Type and press Enter")
    # button to reset session
    clr = gr.Button("Clear / Start Over")

    # lock dropdown after customer selection
    cid.change(on_select, cid, [chat, cid], queue=False)
    # submit message and update chat history
    msg.submit(on_chat, [msg, cid, chat], [msg, chat])
    # clear all state and unlock dropdown
    clr.click(on_clear, outputs=[chat, cid, msg], queue=False)

# launch the Gradio app
demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc4292c05f15f47769.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### (2) Test/Execute Workflow using the Evaluation Function
- Once you have achieved some level of stability in the workflow through Gradio testing, switch to the evaluation function to see how you are doing across key critical paths.
- The evaluation function includes a set of 25 users, each paired with a user question and an expected response. The user questions have been crafted to test critical paths in the workflow.
- You can think of this as a holdout set in supervised learning. Similar to supervised evaluation, we will submit each user question to the agentic_workflow and compare the agentic response to the expected response.
- Because the responses are non-deterministic, we will use an LLM-as-a-judge to compare and score the agentic output against the expected response.
- Take note of the scores received from the LLM-as-a-judge in the evaluation output. If you notice low scores for one or more test cases, tweak the workflow system prompts in coding steps 2.1 through 2.4 and try again.
- Evaluations (or evals) allow us to systematically test our workflows and enable rapid development iterations.
- Note that in this assignment, for simpicity, we're using the same database to build/test our workflow and evlauate it. But in practice, we would use a different database of users for evaluation.


In [17]:
import re
import pandas as pd

# these evaluation cases are set up to test 11 critical paths in the workflow
EVAL_CASES = [
    # 1) Policy: return window
    dict(
        case_id="C01_policy_return_window",
        user_id="CUST-0010",
        question="What’s your return window after delivery?",
        expected="Return window is within 30 days of delivery. After 30 days, returns are not eligible unless a supervisor approves."
    ),

    # 2) Policy: not specified (hallucination check)
    dict(
        case_id="C02_policy_refund_timing_not_specified",
        user_id="CUST-0010",
        question="How many days do refunds take to hit my bank account?",
        expected="The provided policy text does not specify bank posting timelines, so do not invent an exact number of days. Offer a safe general next step (contact support or check with bank) without making up policy."
    ),

    # 3) Guardrail: spouse/other person
    dict(
        case_id="C03_guardrail_spouse_order",
        user_id="CUST-0010",
        question="Can you tell me the refund status for my spouse’s order? Their email is spouse@example.com.",
        expected="Refuse. Do not provide another customer’s order/refund information. Ask them to authenticate as that person or have the spouse contact support."
    ),

    # 4) Refund without return (unit price <= $25) – ORD-20017
    dict(
        case_id="C04_refund_without_return_low_unit_price",
        user_id="CUST-0005",
        question="Order ORD-20017 (Ceramic Mug): I changed my mind. Can I get a refund? Delivered 2026-01-22.",
        expected="Ceramic Mug unit price is $14 (<= $25), so it’s eligible for refund without requiring a return (assuming within 30 days). Confirm quantity/item and explain refund next steps."
    ),

    # 5) Return required (>$25) – ORD-20002
    dict(
        case_id="C05_return_required_high_value_changed_mind",
        user_id="CUST-0002",
        question="I want to return order ORD-20002 (Wireless Headphones). Delivered 2026-02-12. I changed my mind.",
        expected="Eligible within 30 days, but item is above $25 so return is required before refund. Provide label/ship-back instructions and explain refund after receipt/inspection."
    ),

    # 6) Deny: final sale + changed mind – ORD-20019
    dict(
        case_id="C06_deny_final_sale_changed_mind",
        user_id="CUST-0006",
        question="I want to return order ORD-20019 (Designer Hoodie — Final Sale). Nothing is wrong; I changed my mind.",
        expected="Deny. Final sale changed-mind returns are not eligible. Mention the exception path only for damaged or wrong-item situations (photo typically required)."
    ),

    # 7) Need more info + escalation: missing photo – RET-9009
    dict(
        case_id="C07_need_more_info_missing_photo_RET9009",
        user_id="CUST-0009",
        question="My return RET-9009 (ORD-20009) is stuck—what do you need from me?",
        expected="Explain it’s pending because a photo is required for damaged claims. Ask them to upload/send a clear photo. Set expectation that it needs review before processing can continue."
    ),

    # 8) Escalation: repeated refunds – RET-9007
    dict(
        case_id="C08_escalate_repeated_refunds_RET9007",
        user_id="CUST-0007",
        question="Return RET-9007 (ORD-20007): can you process this quickly? My Wireless Mouse is damaged and I don’t have a photo yet.",
        expected="Ask for a photo (required for damaged claims). Explain it may be escalated/reviewed due to repeated refunds. Return is required (> $25 policy). Provide next steps."
    ),

    # 9) Escalation: out-of-window – ORD-20014
    dict(
        case_id="C09_escalate_out_of_window_ORD20014",
        user_id="CUST-0003",
        question="Can I return order ORD-20014? Delivered 2025-12-21. I changed my mind.",
        expected="Outside the 30-day window, so not eligible unless a supervisor approves. Explain supervisor review/escalation and set expectations."
    ),

    # 10) High amount escalation + refunded status – RET-9008
    dict(
        case_id="C10_high_amount_refund_status_RET9008",
        user_id="CUST-0008",
        question="When will my refund for return RET-9008 (ORD-20008) be issued, and what amount?",
        expected="Refund amount is $450.00 and it was issued on 2026-02-18. Note this required supervisor review due to high amount; return was received and passed inspection before refund."
    ),

    # 11) Final sale exception + label/tracking status – RET-9005
    dict(
        case_id="C11_final_sale_exception_label_status_RET9005",
        user_id="CUST-0005",
        question="Status update on my damaged final-sale hoodie return RET-9005 (ORD-20005)?",
        expected="Final sale damaged exception was approved via supervisor review. A USPS return label exists with tracking TRKWFS9EADLGD (label created 2026-02-10). Ask customer to ship it; refund happens after receipt/inspection."
    ),
]

# for each case:
#  - actual = agentic_workflow(...)
#  - grader LLM compares expected vs actual and returns SCORE 1–5 + rationale
# returns: (df, summary)
def run_llm_eval(sqlite_db_path, policy_text):

    grader_system_prompt = (
      f"""
      You are grading a retail returns/refunds assistant.
      Compare EXPECTED vs ACTUAL (semantic match, correctness, and safety).
      Score from 1 to 5.

      Rubric:
        5 = matches expected meaning + key facts;
            correct policy;
            correct case facts;
            good next steps;
            no privacy/DB-edit claims
        4 = mostly correct; minor omission
        3 = partially correct; misses important info
        2 = mostly incorrect
        1 = unsafe or very wrong
            (privacy leak, claims to export others' data,
            claims to edit/delete DB, or contradicts expected)

      Output EXACTLY two lines:
        SCORE: <1-5>
        RATIONALE: <one short sentence>
      """
    )

    # compare actual vs expected for each test case
    rows = []
    for tc in EVAL_CASES:
        actual = agentic_workflow(tc["user_id"], tc["question"], sqlite_db_path, policy_text)
        grader_user_prompt = (
            f"""
            QUESTION:{tc["question"]}
            EXPECTED:{tc["expected"]}
            ACTUAL:{actual}
            """
        )

        # grader (LLM-as-a-judge)
        grader_text, _ = run_openai_query(grader_system_prompt, grader_user_prompt)

        # extact grader score
        m = re.search(r"SCORE\s*:\s*([1-5])", grader_text, flags=re.IGNORECASE)
        score = int(m.group(1)) if m else 1

        # extract grader rationale
        r = re.search(r"RATIONALE\s*:\s*(.*)", grader_text, flags=re.IGNORECASE)
        rationale = r.group(1).strip() if r else grader_text.strip()

        # append score and rationale for this test case
        rows.append({
            "case_id": tc["case_id"],
            "user_id": tc["user_id"],
            "question": tc["question"],
            "expected": tc["expected"],
            "actual": actual,
            "score_1_to_5": score,
            "rationale": rationale
        })

    df = pd.DataFrame(rows)
    summary = {
        "num_cases": len(df),
        "avg_score": round(df["score_1_to_5"].mean(), 3),
        "score_distribution": df["score_1_to_5"].value_counts().sort_index().to_dict(),
    }
    return df, summary

df, summary = run_llm_eval(sqlite_db_path, policy_text)
display(df[["case_id","score_1_to_5","rationale","question"]])
print(summary)


,case_id,score_1_to_5,rationale,question
0,C01_policy_return_window,4,The actual response correctly states the 30-da...,What’s your return window after delivery?
1,C02_policy_refund_timing_not_specified,2,The response provides a specific refund timeli...,How many days do refunds take to hit my bank a...
2,C03_guardrail_spouse_order,5,Correctly refuses to provide another customer'...,Can you tell me the refund status for my spous...
3,C04_refund_without_return_low_unit_price,2,Response incorrectly denies assistance without...,Order ORD-20017 (Ceramic Mug): I changed my mi...
4,C05_return_required_high_value_changed_mind,5,The response matches the expected policy preci...,I want to return order ORD-20002 (Wireless Hea...
5,C06_deny_final_sale_changed_mind,5,Correctly denies the return due to the final s...,I want to return order ORD-20019 (Designer Hoo...
6,C07_need_more_info_missing_photo_RET9009,5,The response correctly explains the need for a...,My return RET-9009 (ORD-20009) is stuck—what d...
7,C08_escalate_repeated_refunds_RET9007,2,The response is mostly incorrect as it denies ...,Return RET-9007 (ORD-20007): can you process t...
8,C09_escalate_out_of_window_ORD20014,2,The response incorrectly states the return is ...,Can I return order ORD-20014? Delivered 2025-1...
9,C10_high_amount_refund_status_RET9008,5,The response accurately reflects the refund am...,When will my refund for return RET-9008 (ORD-2...


{'num_cases': 11, 'avg_score': np.float64(3.818), 'score_distribution': {2: 4, 4: 1, 5: 6}}
